In [1]:
from dotenv import load_dotenv
load_dotenv()

import os
PDF_PATH = "가상Tech_업무가이드.pdf"

if not os.path.exists(PDF_PATH):
    raise FileNotFoundError(f"PDF를 찾을 수 없습니다: {PDF_PATH}")
print(f"PDF 경로: {PDF_PATH}")

PDF 경로: 가상Tech_업무가이드.pdf


In [2]:
from pypdf import PdfReader
from langchain_core.documents import Document

# ---------- PDF 로더 및 Document 객체설정(metadata 기본설정) ----------
def load_pdf_by_page(pdf_path):
    """PDF를 페이지별로, LangChain Document로 변환 """

    # 문서 로딩
    reader = PdfReader(pdf_path)
    
    # documents 초기화
    documents = []

    for page_index, pdf_page in enumerate(reader.pages):
        documents.append(
            Document(
                # 텍스트 추출 예외처리(이미지는 ""로 처리) 
                page_content=pdf_page.extract_text() or "",
                # 메타데이터 설정(문서경로, 페이지index, 페이지번호)
                metadata={
                    "source": pdf_path,
                    "page": page_index,
                    "page_number": page_index + 1,
                },
            )
        )

    return documents

pages = load_pdf_by_page(PDF_PATH)


# ---------- Document 객체설정 추가(metadata 확장) ----------
section_by_page = {
    1: "표지 및 문서 정보", 2: "회사와 업무 원칙",
    3: "계정·기기·정보보안", 4: "업무 도구와 커뮤니케이션",
    5: "문서 작성과 지식 관리", 6: "개발·변경·배포",
    7: "장애 대응과 고객 커뮤니케이션", 8: "휴가·비용·복지",
    9: "FAQ와 연락처",
}
for page in pages:
    page_number = page.metadata["page_number"]
    # 메타데이터 수정.
    page.metadata.update(
        # 페이지 번호
        # page_number=page_number,
        # dict.get(key, default_value): 섹션값으로 번호를 할당하고, 번호가 없으면 "기타"로 할당.
        section=section_by_page.get(page_number, "기타"),
        source_name=os.path.basename(PDF_PATH),
    )
print(f"로드된 페이지: {len(pages)}")

로드된 페이지: 9


In [3]:
from collections import defaultdict

from langchain_text_splitters import RecursiveCharacterTextSplitter

# ---------- 문서 청킹 ----------
splitter = RecursiveCharacterTextSplitter(
    # 최대 청크길이
    chunk_size=800,
    # 문맥 중복 청크길이
    chunk_overlap=120,
    # 청킹 단위(문단>줄바꿈>마침표+공백>다.>요.>띄어쓰기>글자)순으로 나누기
    separators=["\n\n", "\n", ". ", "다.", "요.", " ", ""],
)
chunks = splitter.split_documents(pages)


# ---------- 페이지별 청크 카운팅 및 청크id 생성 ----------

# dict(int) : 키값이 없으면 오류남.
# defaultdict: 키값이 없으면 0으로 설정함.(예외처리 역할)
page_counters = defaultdict(int)
for chunk in chunks:
    page_number = chunk.metadata["page_number"]
    page_counters[page_number] += 1
    # 02d : 두자리 정리 포맷설정
    chunk.metadata["chunk_id"] = f"p{page_number:02d}-c{page_counters[page_number]:02d}"

print(f"생성된 chunk: {len(chunks)}")


# ---------- 출력결과 미리보기 ----------
for chunk in chunks:
    # replace("\n", " ") : 줄바꿈 문자를 공백으로 치환.
    preview = chunk.page_content.replace("\n", " ")[:70]

    # chunk_id 출력, section 출력, [:70] 앞 ~ 70자를 잘라서 출력.
    print(chunk.metadata["chunk_id"], chunk.metadata["section"], preview)

생성된 chunk: 15
p01-c01 표지 및 문서 정보 VIRTUALTECH · INTERNAL HANDBOOK 가상Tech 업무 가이드 함께 일하는 방식부터 보안, 개발, 장애 대
p02-c01 회사와 업무 원칙 가상Tech 업무 가이드 사내용 · v1.0 문서 책임: People Operations 02 CHAPTER 01 회사와 업무
p02-c02 회사와 업무 원칙 집중시간 14:00-16:00 화·목은 사내 정기회의 생성 금지 응답 기대 업무시간 4시간 긴급은 Slack #help-urg
p03-c01 계정·기기·정보보안 가상Tech 업무 가이드 사내용 · v1.0 문서 책임: People Operations 03 CHAPTER 02 계정·기기·
p03-c02 계정·기기·정보보안 신청합니다. • 퇴사·휴직 시 기기는 마지막 근무일 17:00까지 IT 자산 담당자에게 반납합니다.   피싱 신고 의심 메일의
p04-c01 업무 도구와 커뮤니케이션 가상Tech 업무 가이드 사내용 · v1.0 문서 책임: People Operations 04 CHAPTER 03 업무 도구와
p04-c02 업무 도구와 커뮤니케이션 • 긴급이 아닌 업무는 19:00 이후 즉시 응답을 기대하지 않습니다. 예약 전송을 권장합니다.  • 회의 초대에는 목적, 사
p05-c01 문서 작성과 지식 관리 가상Tech 업무 가이드 사내용 · v1.0 문서 책임: People Operations 05 CHAPTER 04 문서 작성과
p06-c01 개발·변경·배포 가상Tech 업무 가이드 사내용 · v1.0 문서 책임: People Operations 06 CHAPTER 05 개발·변경·
p06-c02 개발·변경·배포 • 오류율이 기준 대비 2배 또는 핵심 API 성공률이 99.5% 미만이면 즉시 롤백을 검토합니다.  • 금요일 15:00 이
p07-c01 장애 대응과 고객 커뮤니케이션 가상Tech 업무 가이드 사내용 · v1.0 문서 책임: People Operations 07 CHAPTE

In [4]:
from hashlib import sha256
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

# ---------- PDF Hash기반 DB 경로 설정 ----------

# "rb" : 읽기전용Binary이진데이터(0또는1)[.pdf, .png, .mp4, .zip 등]
with open(PDF_PATH, "rb") as pdf_file:
    # 이진수12자리생성
    pdf_hash = sha256(pdf_file.read()).hexdigest()[:12]
DB_PATH = f"./chroma_db/{pdf_hash}"

# ---------- 컬렉션 네임 설정 ----------
COLLECTION_NAME = "virtualtech-guide"

# ---------- 임베딩 모델 설정 ----------
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")


# ---------- 조건부 DB로드 및 캐싱(If-Else) ----------
if os.path.exists(f"{DB_PATH}/chroma.sqlite3"):
    vectorstore = Chroma(
        collection_name=COLLECTION_NAME,
        persist_directory=DB_PATH,
        embedding_function=embeddings,
    )
    print(f"기존 DB 로드: {DB_PATH}")
else:

    # "chroma.sqlite3" DB가 없는 경우에만 실행함.
    vectorstore = Chroma.from_documents(
        documents=chunks, 
        embedding=embeddings,
        collection_name=COLLECTION_NAME, 
        persist_directory=DB_PATH,
        # p01-c01 형태의 chunk_id를,  DB의 ids=[...]로 지정함.
        ids=[chunk.metadata["chunk_id"] for chunk in chunks],
    )
    print(f"새 DB 생성: {DB_PATH}")

# ---------- 리트리버 (유사도 상위4개 청크를 결과로 전달) ----------
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

새 DB 생성: ./chroma_db/9ed5c24a2a9d


In [ ]:
# ---------- QA 시스템 평가를 위해, 테스트 케이스에 정답 청크(chunk_id)를 매핑 ----------
test_cases = [
    (1, "가상Tech의 핵심 협업시간은 언제인가요?", 2, "1.3 근무 방식", ["핵심 협업시간", "11:00-16:00"], "11:00-16:00입니다."),
    (2, "신입사원이 입사 첫날 18시까지 해야 할 일은 무엇인가요?", 2, "신입 첫날 체크", ["HRIS", "MFA", "버디"], "HRIS, 보안 교육, MFA, 팀 채널, 버디 온보딩을 완료합니다."),
    (3, "계정 탈취가 의심되면 어디로 신고해야 하나요?", 3, "2.1 계정과 인증", ["02-555-0142", "#security-incident"], "네트워크를 끊고 전화 또는 보안 채널로 신고합니다."),
    (4, "고객 원본 데이터는 어떤 등급이며 어떻게 보관하나요?", 3, "2.2 정보 분류", ["민감", "로컬 저장 금지"], "민감 등급이며 암호화하고 로컬 저장하지 않습니다."),
    (5, "Slack에서 결정된 정책은 어디에 기록해야 하나요?", 4, "공식 결정", ["Notion 의사결정 로그", "Jira"], "Notion 의사결정 로그 또는 Jira에 기록합니다."),
    (6, "절차 문서의 다음 검토 주기는 얼마인가요?", 5, "4.1 표준 문서 메타데이터", ["절차 3개월"], "3개월입니다."),
    (7, "기능 개발 브랜치 이름 규칙은 무엇인가요?", 6, "5.1 브랜치와 코드 리뷰", ["feature/JIRA-번호-요약"], "feature/JIRA-번호-요약 형식입니다."),
    (8, "고위험 배포의 승인자와 허용 시간은 무엇인가요?", 6, "5.2 배포 등급과 승인", ["팀장+SRE", "화·목 14:00-16:00"], "팀장과 SRE 승인 후 화·목 14:00-16:00에 진행합니다."),
    (9, "운영 DB 접근 권한은 어떻게 신청하며 언제 회수되나요?", 6, "운영 DB 접근", ["DB-ACCESS", "최대 4시간"], "DB-ACCESS 요청과 승인이 필요하며 최대 4시간 후 회수됩니다."),
    (10, "P1 장애의 호출·상황실·공지 목표 시간은 어떻게 되나요?", 7, "6.1 심각도 기준", ["5분 내 호출", "15분 내 상황실", "30분 주기 공지"], "5분 내 호출, 15분 내 상황실, 30분 주기 공지입니다."),
    (11, "P1 장애 회고는 언제까지 완료해야 하나요?", 7, "6.3 종료와 회고", ["P1은 2영업일"], "2영업일 안에 완료합니다."),
    (12, "성장 지원 복지의 연간 한도와 신청 방법은 무엇인가요?", 8, "7.3 주요 복지", ["연 120만원", "LearnHub"], "연 120만원이며 LearnHub에서 사전 승인합니다."),
    (13, "사용하지 않은 재택근무일을 다음 주로 이월할 수 있나요?", 9, "FAQ", ["이월되지 않습니다"], "아니요. 이월되지 않습니다."),
    (14, "업무용 생성형 AI에 고객 원본 데이터를 입력해도 되나요?", 9, "FAQ", ["고객 원본 데이터", "입력은 금지"], "금지되며 비식별 예시만 허용됩니다."),
    (15, "외부 협력사와 Drive 파일을 공유할 때 조건은 무엇인가요?", 9, "FAQ", ["특정 이메일", "최대 30일"], "승인 후 특정 이메일에 공유하며 만료일은 최대 30일입니다."),
]
# 튜플(Tuple) 구조로 작성된 test_cases 리스트를 Key-Value 형태의 딕셔너리(Dictionary) 리스트로 변환
test_cases = [dict(zip(["id", "question", "page", "section", "anchors", "answer"], row)) for row in test_cases]

# ---------- 정답 청크 매핑 함수 ----------
def answer_chunk_ids(case):

    # 페이지 번호 필터링 : 테스트 케이스의 page 번호와 일치하는 청크만 후보로 선별.
    candidates = [c for c in chunks if c.metadata["page_number"] == case["page"]]

    # 키워드 매칭 : 테스트 케이스에 정의된 키워드 목록(anchors)의 단어 검색후,
    # 청크 본문(page_content)에 몇 개나 포함되어 있는지 대소문자 구분 없이 계산.
    scored = [(sum(a.lower() in c.page_content.lower() for a in case["anchors"]), c) for c in candidates]

    # 최고점수 매칭
    best = max((score for score, _ in scored), default=0)

    # 가장 많은 키워드가 포함된 청크id 목록을 추출.
    #순서1: result = []
    #순서2: for score, c in scored:  
    #순서3:     if score == best and score > 0: 
    #순서4:         result.append(c.metadata["chunk_id"]) 
    #순서5: return result 
    return [c.metadata["chunk_id"] for score, c in scored if score == best and score > 0]

# ---------- 매핑 수행 및 결과 출력 ----------
for case in test_cases:
    case["answer_chunk_ids"] = answer_chunk_ids(case)
    print(f"{case['id']:>2}. p.{case['page']} | {case['section']} | {case['answer_chunk_ids']} | 정답: {case['answer']}")

 1. p.2 | 1.3 근무 방식 | ['p02-c01'] | 정답: 11:00-16:00입니다.
 2. p.2 | 신입 첫날 체크 | ['p02-c02'] | 정답: HRIS, 보안 교육, MFA, 팀 채널, 버디 온보딩을 완료합니다.
 3. p.3 | 2.1 계정과 인증 | ['p03-c01'] | 정답: 네트워크를 끊고 전화 또는 보안 채널로 신고합니다.
 4. p.3 | 2.2 정보 분류 | ['p03-c01'] | 정답: 민감 등급이며 암호화하고 로컬 저장하지 않습니다.
 5. p.4 | 공식 결정 | ['p04-c01', 'p04-c02'] | 정답: Notion 의사결정 로그 또는 Jira에 기록합니다.
 6. p.5 | 4.1 표준 문서 메타데이터 | ['p05-c01'] | 정답: 3개월입니다.
 7. p.6 | 5.1 브랜치와 코드 리뷰 | ['p06-c01'] | 정답: feature/JIRA-번호-요약 형식입니다.
 8. p.6 | 5.2 배포 등급과 승인 | ['p06-c01'] | 정답: 팀장과 SRE 승인 후 화·목 14:00-16:00에 진행합니다.
 9. p.6 | 운영 DB 접근 | ['p06-c02'] | 정답: DB-ACCESS 요청과 승인이 필요하며 최대 4시간 후 회수됩니다.
10. p.7 | 6.1 심각도 기준 | ['p07-c01'] | 정답: 5분 내 호출, 15분 내 상황실, 30분 주기 공지입니다.
11. p.7 | 6.3 종료와 회고 | ['p07-c01', 'p07-c02'] | 정답: 2영업일 안에 완료합니다.
12. p.8 | 7.3 주요 복지 | ['p08-c01'] | 정답: 연 120만원이며 LearnHub에서 사전 승인합니다.
13. p.9 | FAQ | ['p09-c01'] | 정답: 아니요. 이월되지 않습니다.
14. p.9 | FAQ | ['p09-c01'] | 정답: 금지되며 비식별 예시만 허용됩니다.
15. p.9 | FAQ | ['p09-c01'] | 정답: 승인 후 특정 이메일에 공유